In [1]:
import sqlite3 as sq
import pandas as pd

In [2]:
# Load the 2025 Chicago crime dataset, inspect the first few rows, and check column names
crime = pd.read_csv("chicago_crime_2025.csv")
crime.head()
crime.columns

Index(['id', 'case_number', 'date', 'block', 'iucr', 'primary_type',
       'description', 'location_description', 'arrest', 'domestic', 'beat',
       'district', 'ward', 'community_area', 'fbi_code', 'x_coordinate',
       'y_coordinate', 'year', 'updated_on', 'latitude', 'longitude',
       'location'],
      dtype='str')

In [3]:
# Convert 'date' column to datetime
crime["date"] = pd.to_datetime(crime["date"])

# Extract hour and month into new columns
crime["hour"] = crime["date"].dt.hour
crime["month"] = crime["date"].dt.month

# Establish a connection to a SQLite database
conn = sq.connect("crime.db")

In [4]:
# Save the Pandas DataFrame to the SQLite database as table "chicago_crime_2025",
# replacing the table if it already exists and excluding the DataFrame index
crime.to_sql("chicago_crime_2025",conn,if_exists="replace",index=False)

236633

**Query 1 - Window function: rank crime types by frequency**

In [5]:
query = """SELECT primary_type,
       COUNT(*) AS crime_count,
       RANK() OVER (ORDER BY COUNT(*) DESC) AS crime_rank
FROM chicago_crime_2025
GROUP BY primary_type;"""

output = pd.read_sql(query, conn) 
output

,primary_type,crime_count,crime_rank
0,THEFT,55040,1
1,BATTERY,42538,2
2,CRIMINAL DAMAGE,26205,3
3,ASSAULT,21559,4
4,MOTOR VEHICLE THEFT,17232,5
5,OTHER OFFENSE,16742,6
6,DECEPTIVE PRACTICE,14740,7
7,BURGLARY,9710,8
8,NARCOTICS,7417,9
9,ROBBERY,5816,10


**Query 2 - JOIN: Crime totals with arrest rates**

In [6]:
query = """SELECT c.primary_type,
       c.crime_count,
       a.arrest_rate
FROM
    (SELECT primary_type, COUNT(*) AS crime_count
     FROM chicago_crime_2025
     GROUP BY primary_type) c
JOIN
    (SELECT primary_type, AVG(arrest) AS arrest_rate
     FROM chicago_crime_2025
     GROUP BY primary_type) a
ON c.primary_type = a.primary_type
ORDER BY c.crime_count DESC;"""

output = pd.read_sql(query, conn) 
output

,primary_type,crime_count,arrest_rate
0,THEFT,55040,0.089499
1,BATTERY,42538,0.190700
2,CRIMINAL DAMAGE,26205,0.043122
3,ASSAULT,21559,0.124171
4,MOTOR VEHICLE THEFT,17232,0.033658
5,OTHER OFFENSE,16742,0.202365
6,DECEPTIVE PRACTICE,14740,0.030258
7,BURGLARY,9710,0.047271
8,NARCOTICS,7417,0.948092
9,ROBBERY,5816,0.094051


**Query 3 - Crime counts by month for 2025**

In [7]:
query = """SELECT month, COUNT(*) AS crime_count
FROM chicago_crime_2025
GROUP BY month
ORDER BY crime_count DESC;"""

output = pd.read_sql(query, conn) 
output

,month,crime_count
0,7,22624
1,8,21273
2,6,21092
3,10,20898
4,5,20459
5,9,20283
6,3,19741
7,4,19617
8,1,18487
9,11,18312
